In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import model_selection, linear_model, metrics

In [159]:
#!powershell Invoke-WebRequest 'https://drive.google.com/uc?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx' -UseBasicParsing -OutFile MovieLens.zip

In [160]:
#!powershell Expand-Archive -Path MovieLens.zip

In [161]:
links = pd.read_csv('MovieLens/links.csv')
movies = pd.read_csv('MovieLens/movies.csv')
movies = movies.drop(movies[movies.genres == "(no genres listed)"].index).reset_index(drop=True)
ratings = pd.read_csv('MovieLens/ratings.csv')
tags = pd.read_csv('MovieLens/tags.csv')

## Цель: определить какой рейтинг поставит конкрнетному фильму конуретный пользователь

## План
- В таблицу с рейтингами дополняем признаком "Средний рейтинг" (этого фильма среди всех пользователей)
- В таблицу с рейтингами дополняем признаком "Средний рейтинг пользователя" (этого пользователя для всех фильмов)
- Получаем tf-idf по жанрам для каждого фильма
- В таблицу с рейтингами добавляем занчение tf-idf по жанрам, удаляем фильмы, для которых нет жанров.
- Обучим модель и попробуем предсказать рейтинг пользователя по фильму.

- Получаем tf-idf по тэгам для каждого фильма. Игнорируем фильмы без тэгов.
- В таблицу с рейтингами добавляем занчение tf-idf по тэгам, удаляем фильмы, для которых которые нет тэгов.
- Обучим модель и попробуем предсказать рейтинг пользователя по фильму.

## Дополняем признаком "Средний рейтинг"

In [162]:
mean_rating = ratings[["movieId", "rating"]].groupby("movieId").mean().rename(columns={"rating":"mean_rating"})
with_mean_rating = ratings.merge(mean_rating, on="movieId", how="left")
#with_mean_rating

## Дополняем признаком "Средний рейтинг пользователя"

In [163]:
mean_user_rating = ratings[["userId", "rating"]].groupby("userId").mean().rename(columns={"rating":"mean_user_rating"})
with_mean_user_rating = with_mean_rating.merge(mean_user_rating, on="userId", how="left", )
#with_mean_user_rating

## Получаем tf-idf по жанрам для каждого фильма

In [164]:
def normalize_genre(s):
    return s.replace(' ', '').replace('-', '')

movie_genres = [normalize_genre(g) for g in movies.genres.values]

tfidf_vectorizer = TfidfVectorizer()
tfidf_values = tfidf_vectorizer.fit_transform(movie_genres)

tfidf_df_genres = pd.DataFrame(tfidf_values.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df_genres["movieId"] = movies.movieId
#tfidf_df_genres

## В таблицу с рейтингами добавляем занчение tf-idf по жанрам

In [165]:
# Здесь удалим NaN, потому что выше мы удалили фильмы без жанров, т.е. не все строчки с рейтингами найдут свои фильмы
with_genres_tfidf = with_mean_user_rating.merge(tfidf_df_genres, how="left", on="movieId").dropna()
#with_genres_tfidf

## Обучим модель и попробуем предсказать рейтинг пользователя по фильму

In [166]:
X = with_genres_tfidf.drop(columns=["userId", "timestamp", "movieId", "rating"])
Y = with_genres_tfidf["rating"]

X_train, X_test, Y_train, Y_test = model_selection.train_test_split(X, Y, test_size=0.2, random_state=42)

lr = linear_model.LinearRegression().fit(X_train, Y_train)
Y_pred = lr.predict(X_test)

loss = metrics.root_mean_squared_error(Y_test, Y_pred)
loss

0.8078336466440911

## Получаем tf-idf по тэгам для каждого фильма

In [167]:
def normalize_tags(s):
    return s.lower().replace(' ', '').replace('-', '')

user_movie_tags = [normalize_tags(t) for t in tags.tag]

tfidf_vectorizer2 = TfidfVectorizer()
tfidf_values2 = tfidf_vectorizer2.fit_transform(user_movie_tags)

tfidf_df_tags = pd.DataFrame(tfidf_values2.toarray(), columns=tfidf_vectorizer2.get_feature_names_out())
tfidf_df_tags["movieId"] = tags.movieId
tfidf_df_tags["userId"] = tags.userId
#tfidf_df_tags

## В таблицу с рейтингами добавляем занчение tf-idf по тэгам

In [168]:
# Здесь удалим NaN, потому что выше мы удалили фильмы без жанров и у нас много фильмов без тэгов
with_tags_tfidf = with_genres_tfidf.merge(tfidf_df_tags, how="left", on=["movieId", "userId"]).dropna()
#with_tags_tfidf

## Обучим модель и попробуем предсказать рейтинг пользователя по фильму

In [169]:

X2 = with_tags_tfidf.drop(columns=["userId", "timestamp", "movieId", "rating"])
Y2 = with_tags_tfidf["rating"]

X_train2, X_test2, Y_train2, Y_test2 = model_selection.train_test_split(X2, Y2, test_size=0.2, random_state=42)

lr2 = linear_model.LinearRegression().fit(X_train2, Y_train2)
Y_pred2 = lr2.predict(X_test2)

loss2 = metrics.root_mean_squared_error(Y_test2, Y_pred2)
loss2

0.7494972504176862

## Выводы
- По поведению пользователя (его среднему рейтингу) и среднему рейтингу фильма с учетом жанров можно довольно неплохо прогнозировать рейтинг фильма
- Добавление тэгов ухудшает прогноз, их очень много, они довольно уникальны и несистематизированы, их использование сильно уменьшает выборку.